### Data Ingestion

In [29]:
from dotenv import load_dotenv
from langchain_core.documents import Document

In [30]:
doc=Document(
    page_content="This is a test document.",
    metadata={
        "source": "test_source",
        "author": "Hamza Malik",
        "page": 1,
        "date_created": "2024-06-10",
   }
)
doc

Document(metadata={'source': 'test_source', 'author': 'Hamza Malik', 'page': 1, 'date_created': '2024-06-10'}, page_content='This is a test document.')

In [31]:
import os
os.makedirs("../data/text_files", exist_ok=True)


In [32]:
sample_texts={
    "../data/text_files/python.txt": """Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python has a large standard library and a vibrant ecosystem of third-party packages, making it suitable for a wide range of applications, from web development to data science.""",
    "../data/text_files/machine_learning.txt": """Machine learning is a subset of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to learn from and make predictions or decisions based on data. It involves training models on large datasets to recognize patterns and improve performance over time without being explicitly programmed for specific tasks.""",

}

for file_path, content in sample_texts.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")       

Sample text files created successfully.


In [33]:
from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python.txt")
document=loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python.txt'}, page_content='Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python has a large standard library and a vibrant ecosystem of third-party packages, making it suitable for a wide range of applications, from web development to data science.')]


In [34]:
from langchain_community.document_loaders import DirectoryLoader

directory_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding" : "utf-8"},
    show_progress=False
)

document=directory_loader.load()
document

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine learning is a subset of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to learn from and make predictions or decisions based on data. It involves training models on large datasets to recognize patterns and improve performance over time without being explicitly programmed for specific tasks.'),
 Document(metadata={'source': '..\\data\\text_files\\python.txt'}, page_content='Python is a high-level, interpreted programming language known for its simplicity and readability. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python has a large standard library and a vibrant ecosystem of third-party packages, making it suitable for a wide range of applications, from web development to data science.')]

In [35]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

directory_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)

pdf_document=directory_loader.load()
pdf_document

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-05T16:10:43+00:00', 'source': '..\\data\\pdf\\digital_health_research.pdf', 'file_path': '..\\data\\pdf\\digital_health_research.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': 'The Role of Digital Health Technology in Expanding Healthcare Access', 'author': 'Research Report', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-09-05T16:10:43+00:00', 'trapped': '', 'modDate': "D:20260905161043+00'00'", 'creationDate': "D:20260905161043+00'00'", 'page': 0}, page_content='The Role of Digital Health Technology in Expanding\nHealthcare Access\nA Review of Trends, Benefits, and Challenges in Telemedicine and Health Information Systems\nPrepared as a general research overview | September 2026'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-05T16:10:43+00:00', 'source': '..\\d

In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [37]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
chunks=text_splitter.split_documents(pdf_document)

print(f"Total chunks: {len(chunks)}")
print(chunks[0])          
print(chunks[0].page_content)
print(chunks[0].metadata)



Total chunks: 10
page_content='The Role of Digital Health Technology in Expanding
Healthcare Access
A Review of Trends, Benefits, and Challenges in Telemedicine and Health Information Systems
Prepared as a general research overview | September 2026' metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-05T16:10:43+00:00', 'source': '..\\data\\pdf\\digital_health_research.pdf', 'file_path': '..\\data\\pdf\\digital_health_research.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': 'The Role of Digital Health Technology in Expanding Healthcare Access', 'author': 'Research Report', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-09-05T16:10:43+00:00', 'trapped': '', 'modDate': "D:20260905161043+00'00'", 'creationDate': "D:20260905161043+00'00'", 'page': 0}
The Role of Digital Health Technology in Expanding
Healthcare Access
A Review of Trends, Benefits, and Challenges in Telemedicine and Health Information Systems

In [38]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings; 
from dotenv import load_dotenv
load_dotenv()
HUGGING_FACE=os.getenv("HUGGING_FACE")
print('Installed successfully')
embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-mpnet-base-v2",
    task="feature-extraction",
    huggingfacehub_api_token=HUGGING_FACE,
)
# Sirf chunks ke text nikal lo (page_content)
texts = [chunk.page_content for chunk in chunks]

# Sab chunks ka embedding ek sath generate karo
chunk_embeddings = embeddings.embed_documents(texts)

# Check karo
print(f"Total chunks: {len(chunks)}")
print(f"Total embeddings: {len(chunk_embeddings)}")
print(f"Embedding dimension: {len(chunk_embeddings[0])}")
print(chunk_embeddings[0][:10])

Installed successfully


Total chunks: 10
Total embeddings: 10
Embedding dimension: 768
[0.0210757777094841, 0.023808283731341362, -0.0547846183180809, -0.11752501875162125, 0.006005586590617895, 0.0368778258562088, 0.025493580847978592, 0.057643257081508636, 0.013364140875637531, 0.03212064504623413]


In [39]:
load_dotenv()

True

In [40]:
import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.config import Configure, Property, DataType
from dotenv import load_dotenv
import os

# .env se sab variables load karo
load_dotenv()

WEAVIATE_URL = os.getenv("WEAVIATE_URL")
WEAVIATE_API_KEY = os.getenv("WEAVIATE_API_KEY")

# Client banao (connect karo)
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
)

print(client.is_ready())  # True aana chahiye

# Ab collection create karo
collection_name = "Document"

if not client.collections.exists(collection_name):
    client.collections.create(
        name=collection_name,
        vectorizer_config=Configure.Vectorizer.none(),  # aap khud vectors doge
        properties=[
            Property(name="content", data_type=DataType.TEXT),
            Property(name="source", data_type=DataType.TEXT),
        ],
    )

collection = client.collections.get(collection_name)

True


In [41]:
# Chunks aur unke embeddings ko Weaviate collection mein daalo
with collection.batch.dynamic() as batch:
    for chunk, vector in zip(chunks, chunk_embeddings):
        batch.add_object(
            properties={
                "content": chunk.page_content,
                "source": chunk.metadata.get("source", "unknown"),
            },
            vector=vector,
        )

print(f"✅ {len(chunks)} chunks successfully inserted into Weaviate")

c:\Users\user\Desktop\RAG\.venv\Lib\site-packages\weaviate\warnings.py:312: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\user\AppData\Local\Temp\ipykernel_9356\985849650.py:2: ResourceWarning: unclosed <ssl.SSLSocket fd=1508, family=2, type=1, proto=0, laddr=('192.168.100.25', 50230), raddr=('3.123.61.162', 443)>
  with collection.batch.dynamic() as batch:


✅ 10 chunks successfully inserted into Weaviate


In [42]:
# Total objects count check karo
response = collection.aggregate.over_all(total_count=True)
print(f"Total objects in collection: {response.total_count}")

Total objects in collection: 20


In [ ]:
# User se query lo
user_query = input("Write you query:")

# Query ko embed karo (same model use karna zaroori hai jo chunks ke liye use kiya tha)
query_vector = embeddings.embed_query(user_query)

# Weaviate mein similarity search karo
results = collection.query.near_vector(
    near_vector=query_vector,
    limit=5,  # top 5 sabse relevant chunks
    return_metadata=["distance"]  # kitna "close" match hai, wo bhi dikhega
)

# Results print karo
print(f"\n🔍 Query: {user_query}\n")
for i, obj in enumerate(results.objects, 1):
    print(f"--- Result {i} ---")
    print(f"Content: {obj.properties['content'][:300]}...")  # pehle 300 characters
    print(f"Source: {obj.properties['source']}")
    print(f"Distance: {obj.metadata.distance}")
    print()